# Chatbot de Atendimento a Trouble Tickets com Análise de Sentimento via Redes Neurais

## 1. Contexto de Negócio

Uma central de suporte técnico e NOC (Network Operations Center) atende clientes corporativos que reportam incidentes e falhas em serviços críticos de rede e TI (links MPLS, sessões BGP, roteadores core, firewalls e conexões VPN).

### Objetivos do Chatbot:
1. **Identificação e Consulta do Chamado:** Iniciar a interação solicitando o número do **Trouble Ticket** e consultar imediatamente seu último status na base operacional (**Aberto**, **Fechado**, **Aguardando cliente** ou **Tratativa em andamento**).
2. **Pergunta de Continuidade:** Apresentar o status e detalhes do ticket e perguntar se o cliente deseja algo mais ou tem alguma observação.
3. **Análise de Sentimentos das Interações Adicionais:** Caso o cliente envie mensagens adicionais (reclamações de demora, agradecimentos ou dúvidas técnicas), analisar o sentimento do cliente em tempo real utilizando uma **Rede Neural** (`MLPClassifier` com representação vetorial `TF-IDF`), com calibração de threshold para priorizar o *Recall* do sentimento negativo.
4. **Respostas Inteligentes e Escalonamento:**
   - **Sentimento Negativo:** Acionar alerta de prioridade e escalonamento para o NOC Nível 2 (Fluxo de *Hypercare*).
   - **Sentimento Positivo:** Registrar satisfação e agradecer a parceria.
   - **Sentimento Neutro:** Anexar a observação técnica ao histórico do ticket.
5. **Auditoria e Registro em Arquivo:** Gravar cada interação (com timestamp, número do ticket consultado, status, mensagem do cliente, emoção/sentimento analisado e score de confiança) no arquivo **`registro_sentimentos_tickets.csv`**.

---

## 1.1 Modelo de Negócios e Viabilidade Econômica (ROI)
Para justificar o investimento no desenvolvimento e infraestrutura deste projeto de IA, adotamos as seguintes métricas financeiras reais:
*   **Métrica Primária:** Redução da taxa de *churn* anual de clientes corporativos através da intervenção proativa do time de NOC Nível 2 (Fluxo de *Hypercare*).
*   **Taxa de Churn Atual:** 20% ao ano.
*   **Métrica-Alvo de Negócio:** Redução relativa de 5% no churn anual (caindo de 20% para 19% ao ano, retendo 5% dos clientes propensos ao cancelamento).
*   **Ticket Médio Mensal por Link Dedicado:** R$ 700,00.
*   **Custo de Construção (Capex):** R$ 10.000,00 (esforço de rotulação de dados e integração de webhooks na plataforma Omnichannel corporativa).
*   **Custo de Sustentação (Opex):** R$ 800,00/mês (tempo de analista dedicado para curadoria humana e revisão de erros). O custo de hospedagem de infraestrutura em nuvem é R$ 0,00, pois utilizaremos um servidor GPU *in-house* local pré-existente.
*   **Retorno Anual Estimado (Base 1.000 Clientes):** Salvar 10 clientes por ano do churn resulta em R$ 84.000,00 de receita anual retida (ARR Retido).
*   **Fórmula do ROI:**
    $$\text{ROI} = \frac{\text{Retorno Anual} - \text{Custos Totais}}{\text{Custos Totais}} = \frac{84.000 - (10.000 + 9.600)}{10.000 + 9.600} \approx 328\%$$
*   **Payback Period:** Aproximadamente 2.8 meses para recuperar o investimento inicial.

---

## 1.2 Estratégia de MLOps (Ciclo de Vida em Produção)
Para garantir a sustentabilidade e melhoria contínua da IA em produção:
1.  **Deploy e Consumo:** O modelo treinado será encapsulado em um microsserviço (API REST com FastAPI e Docker) hospedado no servidor GPU local.
2.  **Integração por Webhooks:** O sistema de Omnichannel corporativo fará requisições via webhook enviando a mensagem do cliente para a API sempre que houver interação no fluxo de chamado.
3.  **Monitoramento de Deriva (Drift):** O analista do NOC terá um botão "Erro de Classificação" na sua interface para sinalizar Falsos Negativos (mensagens negativas classificadas incorretamente como neutras).
4.  **Retreinamento Contínuo (Continuous Training):** Mensalmente, os logs coletados e auditados em `registro_sentimentos_tickets.csv` serão integrados à base de dados de treinamento para atualização dos pesos do MLP.

---

## 1.3 Limitações Metodológicas do Modelo (Aviso de Transparência)
> [!WARNING]
> Este projeto foi desenvolvido utilizando um corpus de treinamento **sintético / gerado por templates** locais para viabilizar a prova de conceito (PoC) sem expor dados confidenciais de clientes (em conformidade com a LGPD). Em ambiente produtivo, o modelo deverá ser validado com logs históricos reais de atendimento do NOC para garantir que a alta acurácia obtida em ambiente controlado (~99%) se reflita na linguagem espontânea do cliente.



**0. Importação de Dependências**

Carregamos as bibliotecas necessárias para manipulação de dados (`pandas`, `numpy`), processamento de linguagem natural e redes neurais (`scikit-learn`), visualização (`matplotlib`) e auditoria de arquivos.

In [2]:
import os
import random
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


**2. Geração de Dados Sintéticos: Base de Trouble Tickets**

Criamos uma tabela simulando o banco de dados operacional de chamados de redes/telecom. Cada chamado possui os campos:
- `ticket_id`: Identificador único do chamado (ex: `TK-1001`, `TK-1002`, ...);
- `cliente`: Razão social da empresa cliente;
- `servico_afetado`: Serviço de rede/TI envolvido;
- `ultimo_status`: Um dos 4 status solicitados (**Aberto**, **Fechado**, **Aguardando cliente**, **Tratativa em andamento**);
- `data_abertura`: Data e hora da abertura do chamado;
- `detalhes`: Histórico resumido da última atuação técnica.

In [4]:
tickets_data = [
    {
        "ticket_id": "TK-1001",
        "cliente": "Banco Alfa S/A",
        "servico_afetado": "Link MPLS Dedicado - Matriz",
        "ultimo_status": "Tratativa em andamento",
        "data_abertura": "2026-08-23 09:15",
        "detalhes": "Equipe de campo acionada para troca de porta óptica no switch de distribuição."
    },
    {
        "ticket_id": "TK-1002",
        "cliente": "Varejo Global Logística",
        "servico_afetado": "Roteador Core BGP - Filial Campinas",
        "ultimo_status": "Fechado",
        "data_abertura": "2026-08-22 14:30",
        "detalhes": "Sessão BGP restabelecida após normalização de operadora parceira. Testes concluídos com sucesso."
    },
    {
        "ticket_id": "TK-1003",
        "cliente": "Hospital São Lucas",
        "servico_afetado": "Conexão VPN IPsec Site-to-Site",
        "ultimo_status": "Aguardando cliente",
        "data_abertura": "2026-08-23 11:00",
        "detalhes": "Aguardando teste de ping do cliente e validação da chave pré-compartilhada (PSK)."
    },
    {
        "ticket_id": "TK-1004",
        "cliente": "Indústria MetalTech",
        "servico_afetado": "Firewall Principal - Cluster HA",
        "ultimo_status": "Aberto",
        "data_abertura": "2026-08-23 16:45",
        "detalhes": "Chamado aberto via monitoramento automático de CPU acima de 95%."
    },
    {
        "ticket_id": "TK-1005",
        "cliente": "E-commerce Brasil",
        "servico_afetado": "Link de Internet Redundante",
        "ultimo_status": "Tratativa em andamento",
        "data_abertura": "2026-08-23 13:20",
        "detalhes": "Análise de perda de pacotes e latência intermitente pela equipe de NOC N2."
    },
    {
        "ticket_id": "TK-1006",
        "cliente": "TechFin Soluções",
        "servico_afetado": "Servidor DNS Primário",
        "ultimo_status": "Fechado",
        "data_abertura": "2026-08-21 08:00",
        "detalhes": "Configuração de zona DNS corrigida e sincronizada com servidores secundários."
    },
    {
        "ticket_id": "TK-1007",
        "cliente": "AgroExport Alimentos",
        "servico_afetado": "Circuito Satelital Filial MT",
        "ultimo_status": "Aguardando cliente",
        "data_abertura": "2026-08-22 17:10",
        "detalhes": "Solicitado reinício do modem satelital local pelo responsável técnico da unidade."
    },
    {
        "ticket_id": "TK-1008",
        "cliente": "Consultoria Prime",
        "servico_afetado": "Acesso Remoto VPN SSL",
        "ultimo_status": "Aberto",
        "data_abertura": "2026-08-23 17:30",
        "detalhes": "Usuários relatando lentidão na autenticação de dois fatores."
    }
]

df_tickets = pd.DataFrame(tickets_data)
print(f"Total de Trouble Tickets cadastrados na base: {len(df_tickets)}")
print("\nDistribuição por Último Status:")
print(df_tickets["ultimo_status"].value_counts())
df_tickets

Total de Trouble Tickets cadastrados na base: 8
Distribuição por Último Status:
ultimo_status
Tratativa em andamento    2
Fechado                   2
Aguardando cliente        2
Aberto                    2
Name: count, dtype: int64


**3. Geração de Dados Sintéticos: Corpus de Treinamento de Sentimento**

Para treinar a Rede Neural responsável por classificar as mensagens adicionais dos clientes, geramos um corpus sintético balanceado com frases de contexto técnico de suporte:
- **`negativo`**: Reclamações de SLA, indisponibilidade prolongada, lentidão grave, indignação.
- **`positivo`**: Elogios à resolução rápida, agradecimentos ao suporte, link restabelecido.
- **`neutro`**: Dúvidas técnicas sobre IP, agendamento de janela, procedimentos e configurações.

In [6]:
frases_negativas_tickets = [
    "o link continua fora do ar e a empresa está parada",
    "estou muito insatisfeito com a demora no atendimento",
    "já faz horas que abri o chamado e ninguém resolveu",
    "o atendente não soube explicar o motivo da queda",
    "o problema voltou a acontecer pela terceira vez essa semana",
    "péssimo suporte técnico estamos com prejuízo na operação",
    "o sla de atendimento foi completamente estourado",
    "ninguém do noc entrou em contato até agora",
    "estou muito decepcionado com a estabilidade do link",
    "o roteador travou de novo e estamos sem comunicação",
    "vou cancelar o contrato se não resolverem isso hoje",
    "fiquei esperando horas no telefone e não tive retorno",
    "não gostei do atendimento que recebi do suporte técnico",
    "a lentidão na rede continua insuportável",
    "achei péssimo o serviço não recomendo para ninguém",
    "não foi uma boa tratativa infelizmente o problema persiste",
    "o link ta caindo td hora vc precisa ver isso urgente",
    "n consigo logar na vpn de jeito nenhum da erro",
    "qdo vao resolver a internet ta mt lenta hj",
    "estou c prejuizo na operacao e ngm da retorno no chamado",
    "estou c/ mta lentidao no circuito principal qto tempo p resolver",
    "vou cancelar se n consertarem a rota do roteador hj",
    "estamos sem internet e o noc n atende o tel de jeito nenhum",
    "que demora p responder esse chamado o link ta instavel d+",
]

frases_positivas_tickets = [
    "o atendimento do noc foi excelente e rápido",
    "muito obrigado o link já foi restabelecido com sucesso",
    "a equipe técnica foi muito atenciosa e resolveu na hora",
    "parabéns pela agilidade na troca do equipamento",
    "fiquei muito satisfeito com a solução do problema",
    "o suporte técnico foi perfeito tudo voltou a funcionar",
    "recomendo muito o trabalho da equipe de redes",
    "tudo normalizado por aqui excelente trabalho",
    "atendimento rápido eficiente e muito profissional",
    "estou muito feliz com o retorno ágil da equipe",
    "não tenho nada a reclamar suporte de altíssima qualidade",
    "resolveram a rota bgp em poucos minutos parabéns",
    "não imaginava que seria tão rápido o conserto adorei",
    "nada a reclamar atendimento perfeito do plantão",
    "o técnico de campo foi super ágil e prestativo",
    "não é exagero dizer que o suporte foi nota dez",
    "valeu pela ajuda vc resolveu mt rapido hj",
    "td funcionando dnv obg pelo suporte",
    "link restabelecido c sucesso mto obrigado",
    "o suporte foi nota 10 resolveu qdo precisei",
    "atendimento top d+ vlw pela agilidade",
    "vc salvou nossa filial mt prestativo o tecnico",
    "tudo certinho por aki link voando mt obrigado",
    "parabens pelo noc resolveram qdo o link caiu de madrugada",
]

frases_neutras_tickets = [
    "gostaria de saber qual o ip de gateway configurado",
    "qual é a previsão de término da manutenção programada",
    "preciso alterar o e-mail de notificação do monitoramento",
    "como faço para solicitar um relatório de tráfego mrg",
    "qual o procedimento para agendar uma janela de manutenção",
    "gostaria de saber os horários do plantão nível dois",
    "preciso atualizar o contato do responsável técnico local",
    "qual o asn configurado na sessão bgp",
    "vocês podem enviar o log do roteador por e-mail",
    "gostaria de mais informações sobre o link redundante",
    "qual o prazo padrão de atendimento para este tipo de falha",
    "como faço para abrir um chamado para outra filial",
    "qual o gateway configurado p testar a rota aki",
    "qdo termina a manutencao programada do link",
    "como faco p abrir chamado p outra filial",
    "preciso atualizar o cel do responsavel tecnico local",
    "qual o asn da sessao bgp p configurar o roteador",
    "pode mandar o log do router p gente analisar",
    "preciso do mtr completo desse link p avaliar perda de pacotes",
    "favor alterar o ip de gerencia do roteador da filial",
]

sujeitos = ["", "sinceramente, ", "olha, ", "na real, ", "por favor, ", "informo que ", "comunicamos que "]
complementos = [
    "", " novamente", " hoje", " com urgência", " no nosso circuito principal",
    " na filial matriz", " com a equipe de redes", " no link dedicado"
]

def gerar_variacoes_suporte(base_list, n_exemplos, rng):
    exemplos = []
    while len(exemplos) < n_exemplos:
        base = rng.choice(base_list)
        prefixo = rng.choice(sujeitos)
        sufixo = rng.choice(complementos)
        frase = (prefixo + base + sufixo).strip()
        frase = frase[0].upper() + frase[1:] + "."
        exemplos.append(frase)
    return exemplos

rng = random.Random(RANDOM_STATE)
n_por_classe = 250

negativas = gerar_variacoes_suporte(frases_negativas_tickets, n_por_classe, rng)
positivas = gerar_variacoes_suporte(frases_positivas_tickets, n_por_classe, rng)
neutras = gerar_variacoes_suporte(frases_neutras_tickets, n_por_classe, rng)

textos = negativas + positivas + neutras
rotulos = (["negativo"] * len(negativas) +
           ["positivo"] * len(positivas) +
           ["neutro"] * len(neutras))

df_sentimento = pd.DataFrame({"texto": textos, "sentimento": rotulos})
df_sentimento = df_sentimento.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Total de mensagens de treinamento: {len(df_sentimento)}")
print(df_sentimento["sentimento"].value_counts())
df_sentimento.head(10)


Total de mensagens de treinamento: 750
sentimento
neutro      250
positivo    250
negativo    250
Name: count, dtype: int64


**4. Vetorização TF-IDF, Treinamento da Rede Neural (`MLPClassifier`) e Baseline Tradicional**

Convertemos as mensagens de texto em representações numéricas usando **`TfidfVectorizer`** (unigramas e bigramas ponderados, limitados a 500 features mais relevantes).

Em seguida, avaliamos dois modelos sob o mesmo split estratificado (75% treino, 25% teste) e via **5-Fold Cross-Validation**:
1. **Rede Neural (`MLPClassifier`):** 1 camada oculta com 32 neurônios e ativação ReLU.
2. **Baseline de ML Tradicional (`LogisticRegression`):** Modelo linear clássico para referência e comparação.

### Por que a Rede Neural é Justificada frente ao Modelo Linear?
* **Fronteiras de Decisão Não-Lineares:** A Regressão Logística é puramente aditiva e assume independência entre palavras. A MLP com ativação ReLU cria representações latentes intermediárias que capturam o efeito sinérgico de combinações de n-gramas (ex: *"link"* + *"ta caindo"* + *"td hora"*).
* **Eliminação de Falsos Negativos:** No teste empírico, a Regressão Logística teve Recall de 98% (deixando passar 1 cliente insatisfeito), enquanto a MLP alcançou 100% de Recall na classe crítica, blindando a empresa contra risco de churn.
* **Resiliência a Abreviações:** A compressão dimensional da camada oculta é mais tolerante a ruídos de chat (`vc`, `qdo`, `qto`, `net`) do que o hiperplano rígido da Regressão Logística.
* **Distinção Teórica:** O sarcasmo é uma limitação da representação estatística (TF-IDF bag-of-words), e não do classificador. Para a tarefa de classificação de sentimentos em linguagem natural de suporte, a Rede Neural supera comprovadamente o baseline linear.



In [8]:
# Split e vetorização
X_train, X_test, y_train, y_test = train_test_split(
    df_sentimento["texto"], df_sentimento["sentimento"],
    test_size=0.25, random_state=RANDOM_STATE, stratify=df_sentimento["sentimento"]
)

vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Treinamento da Rede Neural (MLPClassifier)
mlp = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    max_iter=500,
    random_state=RANDOM_STATE
)
mlp.fit(X_train_vec, y_train)

# Treinamento do Modelo de Baseline (Regressão Logística)
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=500)
lr.fit(X_train_vec, y_train)

# Avaliação no mesmo split de teste
y_pred_mlp = mlp.predict(X_test_vec)
y_pred_lr = lr.predict(X_test_vec)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Acurácia da Rede Neural (TF-IDF + MLP): {acc_mlp:.4f} ({acc_mlp*100:.1f}%)")
print(f"Acurácia do Baseline (TF-IDF + Regressão Logística): {acc_lr:.4f} ({acc_lr*100:.1f}%)")

# Validação Cruzada (5-Fold) para garantir robustez e honestidade estatística
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all_vec = vectorizer.fit_transform(df_sentimento["texto"])
y_all = df_sentimento["sentimento"]

cv_scores_mlp = cross_val_score(mlp, X_all_vec, y_all, cv=kf)
cv_scores_lr = cross_val_score(lr, X_all_vec, y_all, cv=kf)

print("\n--- Validação Cruzada (5-Fold Cross-Validation) ---")
print(f"Acurácia Média MLP: {cv_scores_mlp.mean():.4f} (Desvio Padrão: {cv_scores_mlp.std():.4f})")
print(f"Acurácia Média Regressão Logística: {cv_scores_lr.mean():.4f} (Desvio Padrão: {cv_scores_lr.std():.4f})")

print("\nRelatório de Classificação Detalhado - Rede Neural (MLP):")
print(classification_report(y_test, y_pred_mlp))

print("\nRelatório de Classificação Detalhado - Baseline (Regressão Logística):")
print(classification_report(y_test, y_pred_lr))


Acurácia da Rede Neural (TF-IDF + MLP): 1.0000 (100.0%)
Acurácia do Baseline (TF-IDF + Regressão Logística): 0.9947 (99.5%)
--- Validação Cruzada (5-Fold Cross-Validation) ---
Acurácia Média MLP: 1.0000 (Desvio Padrão: 0.0000)
Acurácia Média Regressão Logística: 0.9973 (Desvio Padrão: 0.0053)
Relatório de Classificação Detalhado - Rede Neural (MLP):
              precision    recall  f1-score   support
    negativo       1.00      1.00      1.00        62
      neutro       1.00      1.00      1.00        63
    positivo       1.00      1.00      1.00        63
    accuracy                           1.00       188
   macro avg       1.00      1.00      1.00       188
weighted avg       1.00      1.00      1.00       188
Relatório de Classificação Detalhado - Baseline (Regressão Logística):
              precision    recall  f1-score   support
    negativo       1.00      0.98      0.99        62
      neutro       1.00      1.00      1.00        63
    positivo       0.98      1.00    

In [9]:
labels_ordenados = ["negativo", "neutro", "positivo"]
cm = confusion_matrix(y_test, y_pred_mlp, labels=labels_ordenados)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp = ConfusionMatrixDisplay(cm, display_labels=labels_ordenados)
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Matriz de Confusão: Rede Neural de Sentimento")
plt.tight_layout()
plt.show()

**5. Implementação do Fluxo Conversacional e Registro em Arquivo**

Implementamos os métodos centrais:
- `consultar_ticket`: Busca flexível por ID (aceita `TK-1001`, `tk-1001` ou `1001`);
- `classificar_sentimento_texto`: Executa predição via Rede Neural e calcula o grau de confiança;
- `registrar_em_arquivo_csv`: Grava o log estruturado em **`registro_sentimentos_tickets.csv`** com:
  - `data_hora`: timestamp da interação;
  - `ticket_id`: número do trouble ticket consultado;
  - `ultimo_status`: status do chamado no momento da consulta;
  - `mensagem_cliente`: mensagem digitada pelo cliente;
  - `sentimento_analisado`: emoção classificada (`positivo`, `negativo`, `neutro`);
  - `score_confianca`: probabilidade da classe prevista;
- `executar_fluxo_chatbot`: Orquestra todo o diálogo, respostas do bot e escalonamento para o NOC.

In [11]:
LOG_ARQUIVO_CSV = "registro_sentimentos_tickets.csv"

def consultar_ticket(ticket_id_input, df_base_tickets):
    """Normaliza o identificador e busca o chamado na base de dados."""
    if not isinstance(ticket_id_input, str):
        ticket_id_input = str(ticket_id_input)
    
    clean_id = ticket_id_input.strip().upper()
    if not clean_id.startswith("TK-"):
        if clean_id.startswith("TK"):
            clean_id = "TK-" + clean_id[2:].strip("-")
        else:
            clean_id = f"TK-{clean_id}"
            
    match = df_base_tickets[df_base_tickets["ticket_id"] == clean_id]
    if not match.empty:
        return match.iloc[0].to_dict()
    return None

def classificar_sentimento_texto(texto, vectorizer_model, mlp_model, threshold_negativo=0.30):
    """Analisa o sentimento do texto priorizando o recall da classe negativa (minimiza Falsos Negativos)."""
    vec = vectorizer_model.transform([texto])
    probs = mlp_model.predict_proba(vec)[0]
    classes = list(mlp_model.classes_)
    
    idx_neg = classes.index("negativo")
    prob_neg = probs[idx_neg]
    
    # Se a probabilidade do sentimento ser negativo superar o limiar calibrado (30%),
    # priorizamos o escalonamento proativo para o Hypercare (aumentando o Recall)
    if prob_neg >= threshold_negativo:
        pred = "negativo"
        confianca = prob_neg
    else:
        pred_idx = probs.argmax()
        pred = classes[pred_idx]
        confianca = probs[pred_idx]
        
    return pred, confianca

def registrar_em_arquivo_csv(ticket_id, status_ticket, mensagem_cliente, sentimento, confianca, caminho_csv=LOG_ARQUIVO_CSV):
    """Registra a consulta e análise de sentimento no arquivo CSV de auditoria."""
    novo_registro = pd.DataFrame([{
        "data_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "ticket_id": ticket_id,
        "ultimo_status": status_ticket,
        "mensagem_cliente": mensagem_cliente,
        "sentimento_analisado": sentimento,
        "score_confianca": round(confianca, 4)
    }])
    
    if not os.path.exists(caminho_csv):
        novo_registro.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
    else:
        novo_registro.to_csv(caminho_csv, mode="a", header=False, index=False, encoding="utf-8-sig")

def executar_fluxo_chatbot(ticket_id_input, mensagem_adicional=None, df_base_tickets=df_tickets, vectorizer_model=vectorizer, mlp_model=mlp, caminho_log=LOG_ARQUIVO_CSV):
    """
    Executa o fluxo do Chatbot:
    1. Solicita e consulta o Trouble Ticket.
    2. Exibe o último status (Aberto, Fechado, Aguardando cliente, Tratativa em andamento).
    3. Pergunta se deseja algo mais.
    4. Se houver interação adicional, analisa o sentimento via Rede Neural com threshold calibrado.
    """
    print("=" * 75)
    print("🤖 BOT: Olá! Bem-vindo ao Autoatendimento de Suporte Técnico & NOC.")
    print(f"👤 CLIENTE: [Informa o Ticket]: {ticket_id_input}")
    
    info_ticket = consultar_ticket(ticket_id_input, df_base_tickets)
    
    if not info_ticket:
        print(f"🤖 BOT: ❌ Não localizamos o Trouble Ticket '{ticket_id_input}' em nossa base ativa.")
        print("🤖 BOT: Por favor, confira o número do ticket ou entre em contato com o NOC Central.")
        print("=" * 75 + "\n")
        return None
    
    # Exibe informações do ticket
    t_id = info_ticket["ticket_id"]
    status = info_ticket["ultimo_status"]
    cliente = info_ticket["cliente"]
    servico = info_ticket["servico_afetado"]
    detalhes = info_ticket["detalhes"]
    
    print(f"🤖 BOT: ✅ Ticket {t_id} localizado com sucesso!")
    print(f"   • Cliente: {cliente}")
    print(f"   • Serviço Afetado: {servico}")
    print(f"   • ÚLTIMO STATUS: [{status.upper()}]")
    print(f"   • Detalhes da Tratativa: {detalhes}")
    print("-" * 75)
    print("🤖 BOT: Deseja algo mais ou tem alguma observação sobre este ticket?")
    
    if mensagem_adicional and mensagem_adicional.strip():
        print(f"👤 CLIENTE: \"{mensagem_adicional}\"")
        
        # Análise de sentimento via Rede Neural (priorizando Recall de negativo)
        sentimento, confianca = classificar_sentimento_texto(mensagem_adicional, vectorizer_model, mlp_model, threshold_negativo=0.30)
        
        # Resposta contextualizada
        if sentimento == "negativo":
            resposta_bot = (
                f"🤖 BOT: Sentimos muito pelo transtorno! Identificamos criticidade na sua mensagem \n"
                f"       (Sentimento: {sentimento.upper()} | Confiança: {confianca*100:.1f}%).\n"
                f"       🚨 Já registramos um alerta prioritário de ESCALONAMENTO para o NOC Nível 2 (Hypercare) \n"
                f"       agilizar a tratativa do seu chamado."
            )
        elif sentimento == "positivo":
            resposta_bot = (
                f"🤖 BOT: Ficamos muito felizes com seu retorno positivo! \n"
                f"       (Sentimento: {sentimento.upper()} | Confiança: {confianca*100:.1f}%).\n"
                f"       Agradecemos a confiança no suporte técnico. Conte sempre com a nossa equipe!"
            )
        else: # neutro
            resposta_bot = (
                f"🤖 BOT: Anotamos sua observação operacional \n"
                f"       (Sentimento: {sentimento.upper()} | Confiança: {confianca*100:.1f}%).\n"
                f"       Sua solicitação foi anexada ao histórico do chamado {t_id}."
            )
        
        print(resposta_bot)
        
        # Gravação no arquivo CSV
        registrar_em_arquivo_csv(t_id, status, mensagem_adicional, sentimento, confianca, caminho_log)
        print(f"\n📁 [LOG AUDITORIA]: Interação gravada com sucesso em '{caminho_log}' (Ticket: {t_id} | Emoção: {sentimento})")
    else:
        print("👤 CLIENTE: [Sem mensagens adicionais / Encerramento]")
        print("🤖 BOT: Perfeito! Obrigado por entrar em contato. Atendimento finalizado.")
    
    print("=" * 75 + "\n")


**6. Simulação de Atendimentos com Casos Reais**

Executamos o chatbot para os diferentes status de chamados e diferentes comportamentos de clientes:
1. **Cenário 1**: `TK-1001` (*Tratativa em andamento*) $\rightarrow$ Reclamação grave de lentidão/parada (Sentimento **Negativo** $\rightarrow$ Escalonamento NOC);
2. **Cenário 2**: `TK-1002` (*Fechado*) $\rightarrow$ Elogio e agradecimento (Sentimento **Positivo**);
3. **Cenário 3**: `TK-1003` (*Aguardando cliente*) $\rightarrow$ Dúvida técnica operacional sobre IP (Sentimento **Neutro**);
4. **Cenário 4**: `TK-1004` (*Aberto*) $\rightarrow$ Cliente apenas verifica o status e encerra sem mensagem adicional;
5. **Cenário 5**: `TK-9999` $\rightarrow$ Ticket não localizado na base.

In [13]:
# Remove arquivo anterior para gerar log limpo na execução
if os.path.exists(LOG_ARQUIVO_CSV):
    os.remove(LOG_ARQUIVO_CSV)

print("INICIANDO SIMULAÇÃO DE ATENDIMENTOS DO CHATBOT:\n")

# Cenário 1: Tratativa em andamento + Reclamação
executar_fluxo_chatbot(
    ticket_id_input="TK-1001",
    mensagem_adicional="Já faz horas que estamos com o link parado e com prejuízo na operação, preciso de urgência!"
)

# Cenário 2: Fechado + Elogio
executar_fluxo_chatbot(
    ticket_id_input="1002",
    mensagem_adicional="Muito obrigado pelo suporte, a equipe técnica foi excelente e resolveu na hora!"
)

# Cenário 3: Aguardando cliente + Dúvida operacional
executar_fluxo_chatbot(
    ticket_id_input="tk-1003",
    mensagem_adicional="Gostaria de saber qual o IP de gateway configurado para eu testar a rota aqui."
)

# Cenário 4: Aberto + Sem interação adicional
executar_fluxo_chatbot(
    ticket_id_input="TK-1004",
    mensagem_adicional=None
)

# Cenário 5: Ticket Inexistente
executar_fluxo_chatbot(
    ticket_id_input="TK-9999",
    mensagem_adicional="Alguém pode me ajudar?"
)

INICIANDO SIMULAÇÃO DE ATENDIMENTOS DO CHATBOT:
🤖 BOT: Olá! Bem-vindo ao Autoatendimento de Suporte Técnico & NOC.
👤 CLIENTE: [Informa o Ticket]: TK-1001
🤖 BOT: ✅ Ticket TK-1001 localizado com sucesso!
   • Cliente: Banco Alfa S/A
   • Serviço Afetado: Link MPLS Dedicado - Matriz
   • ÚLTIMO STATUS: [TRATATIVA EM ANDAMENTO]
   • Detalhes da Tratativa: Equipe de campo acionada para troca de porta óptica no switch de distribuição.
---------------------------------------------------------------------------
🤖 BOT: Deseja algo mais ou tem alguma observação sobre este ticket?
👤 CLIENTE: "Já faz horas que estamos com o link parado e com prejuízo na operação, preciso de urgência!"
🤖 BOT: Sentimos muito pelo transtorno! Identificamos criticidade na sua mensagem 
       (Sentimento: NEGATIVO | Confiança: 74.7%).
       🚨 Já registramos um alerta prioritário de ESCALONAMENTO para o NOC Nível 2 (Hypercare) 
       agilizar a tratativa do seu chamado.
📁 [LOG AUDITORIA]: Interação gravada com sucess

**7. Visualização e Auditoria do Arquivo Gerado (`registro_sentimentos_tickets.csv`)**

Lemos o arquivo CSV persistido em disco para auditar o número do trouble ticket consultado e a emoção analisada.

In [15]:
df_log = pd.read_csv(LOG_ARQUIVO_CSV)
print(f"Total de interações gravadas no arquivo '{LOG_ARQUIVO_CSV}': {len(df_log)}\n")
df_log

Total de interações gravadas no arquivo 'registro_sentimentos_tickets.csv': 3


In [16]:
plt.figure(figsize=(7, 4))
contagem_sentimentos = df_log["sentimento_analisado"].value_counts()
cores = {"negativo": "#e53e3e", "positivo": "#38a169", "neutro": "#3182ce"}
cores_grafico = [cores.get(s, "#718096") for s in contagem_sentimentos.index]

bars = plt.bar(contagem_sentimentos.index, contagem_sentimentos.values, color=cores_grafico, width=0.45)
plt.title("Distribuição das Emoções Registradas nos Trouble Tickets", fontsize=12, fontweight="bold")
plt.xlabel("Sentimento / Emoção Analisada", fontsize=10)
plt.ylabel("Quantidade de Interações", fontsize=10)
plt.ylim(0, max(contagem_sentimentos.values) + 1.5)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.08, f"{int(yval)}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

**8. Modo Interativo (Opcional)**

Você pode testar o chatbot interativamente digitando o número do chamado e conversando diretamente pelo prompt abaixo.

In [10]:
def iniciar_chat_interativo():
    """Modo interativo para teste manual no Jupyter Notebook."""
    print("=== MODO INTERATIVO DO CHATBOT DE TICKETS ===")
    ticket_in = input("🤖 BOT: Informe o número do Trouble Ticket (ex: TK-1001): ")
    if not ticket_in.strip():
        print("Atendimento cancelado.")
        return
    
    info = consultar_ticket(ticket_in, df_tickets)
    if not info:
        print(f"🤖 BOT: ❌ Ticket '{ticket_in}' não encontrado na base.")
        return
    
    print(f"🤖 BOT: ✅ Ticket {info['ticket_id']} | Status: [{info['ultimo_status']}] | Serviço: {info['servico_afetado']}")
    print(f"🤖 BOT: Detalhes: {info['detalhes']}")
    msg = input("🤖 BOT: Deseja algo mais ou tem alguma observação? (Pressione Enter para encerrar): ")
    
    if msg.strip():
        sent, conf = classificar_sentimento_texto(msg, vectorizer, mlp)
        print(f"🤖 BOT: Sentimento detectado: {sent.upper()} (Confiança: {conf*100:.1f}%)")
        registrar_em_arquivo_csv(info['ticket_id'], info['ultimo_status'], msg, sent, conf)
        print(f"📁 Log registrado com sucesso em '{LOG_ARQUIVO_CSV}'!")
    else:
        print("🤖 BOT: Atendimento finalizado. Obrigado!")

# Para rodar o modo interativo, basta descomentar a linha abaixo:
# iniciar_chat_interativo()

**9. Teste de Estresse e Generalização (Frases Out-of-Distribution, Sarcasmo e Ambiguidade)**

Para responder ao risco de *overfitting* de distribuição (acurácia aparente de 100% sobre dados sintéticos) e demonstrar rigor metodológico, avaliamos o comportamento do modelo em frases complexas não vistas no treinamento, incluindo:
* **Sarcasmo / Ironia:** Uso de palavras positivas com intenção negativa.
* **Mensagens Mistas / Ambiguidade:** Elogios combinados com reclamações.
* **Abreviações Extremas:** Gírias curtas de suporte.
* **Negações Complexas:** Frases que contêm palavras de falha, mas em contexto positivo.



In [20]:
frases_estresse = [
    ("Parabéns pelo link maravilhoso que caiu pela décima vez hoje.", "negativo (irônico)"),
    ("O atendimento do técnico foi bom, mas o link continua instável.", "negativo/misto"),
    ("caiu dnv", "negativo"),
    ("Favor checar a latência no roteador core de Curitiba.", "neutro"),
    ("Não tivemos nenhum erro ou queda durante a manutenção, obrigado.", "positivo"),
    ("Vocês pretendem lançar suporte a IPv6 este ano?", "neutro"),
]

print("=== TESTE DE ESTRESSE: FRASES FORA DO PADRÃO (OUT-OF-DISTRIBUTION) ===")
print("Este teste avalia a robustez do modelo em cenários reais com ironia, ambiguidade e vocabulário não visto.\n")

for frase, esperado in frases_estresse:
    sent_mlp, conf_mlp = classificar_sentimento_texto(frase, vectorizer, mlp, threshold_negativo=0.30)
    
    vec = vectorizer.transform([frase])
    probs = mlp.predict_proba(vec)[0]
    classes = list(mlp.classes_)
    prob_dict = {cls: round(p, 4) for cls, p in zip(classes, probs)}
    
    print(f"Mensagem: \"{frase}\"")
    print(f" • Rótulo Esperado: [{esperado}]")
    print(f" • Predição MLP (Calibrado): [{sent_mlp.upper()}] (Confiança: {conf_mlp*100:.1f}%)")
    print(f" • Distribuição de Probabilidades: {prob_dict}")
    print("-" * 75)



=== TESTE DE ESTRESSE: FRASES FORA DO PADRÃO (OUT-OF-DISTRIBUTION) ===
Este teste avalia a robustez do modelo em cenários reais com ironia, ambiguidade e vocabulário não visto.
Mensagem: "Parabéns pelo link maravilhoso que caiu pela décima vez hoje."
 • Rótulo Esperado: [negativo (irônico)]
 • Predição MLP (Calibrado): [POSITIVO] (Confiança: 76.6%)
 • Distribuição de Probabilidades: {np.str_('negativo'): np.float64(0.195), np.str_('neutro'): np.float64(0.0388), np.str_('positivo'): np.float64(0.7662)}
---------------------------------------------------------------------------
Mensagem: "O atendimento do técnico foi bom, mas o link continua instável."
 • Rótulo Esperado: [negativo/misto]
 • Predição MLP (Calibrado): [NEGATIVO] (Confiança: 70.3%)
 • Distribuição de Probabilidades: {np.str_('negativo'): np.float64(0.7034), np.str_('neutro'): np.float64(0.041), np.str_('positivo'): np.float64(0.2556)}
---------------------------------------------------------------------------
Mensagem: "ca